In [67]:
# принудительно перезагрузить модуль
import importlib
import df_preprocessing

importlib.reload(df_preprocessing)

# теперь импортируем функции заново
from df_preprocessing import add_original_to_extended

In [68]:
import os
import re
import ast


from collections import defaultdict
import copy

import numpy as np
import pandas as pd
from IPython.display import display
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt

from df_preprocessing import (
    load_repair_parts,
    extract_model_case_insensitive,
    extract_articles,
    add_original_to_extended,
    find_all_analogs,
    merge_parts_df1,
    merge_parts_df2,
    prepare_sales,
    normalize,
    normalize_analog_lists,
    audit_analog_groups,
    export_to_excel,
    find_mixed_groups,
    article_forms
)

In [69]:
file_csv = r'Запчасти списанные в ремонт.csv'
file_excel = r'Запчасти списанные в ремонт.xlsx'
data = load_repair_parts(file_csv, file_excel)

data = data[data['Номенклатура.Артикул'].notna()]
data = data.drop(columns=['Unnamed: 1', 'Unnamed: 5', 'Unnamed: 6'])
data = data.iloc[:-1].reset_index(drop=True)
data = data[~(data['Номенклатура.Оригинальный номер'].isna() & data['Номенклатура.Артикул'].isna())]

conditions1 = [
    data['Машина'].str.contains('ножничный', case=False, na=False),
    data['Машина'].str.contains('коленчатый', case=False, na=False),
    data['Машина'].str.contains('телескопический', case=False, na=False),
    data['Машина'].str.contains('мачтовый', case=False, na=False)
]

choices1 = ['ножничный', 'коленчатый', 'телескопический', 'мачтовый']

data['Тип подъемника'] = np.select(conditions1, choices1, default='другое')

conditions2 = [

    data['Машина'].str.contains('электрический', case=False, na=False),
    data['Машина'].str.contains('дизельный', case=False, na=False),
]

choices2 = ['Электрический', 'Дизельный']

data['Тип двигателя'] = np.select(conditions2, choices2, default='другое')

data = data.loc[~(data['Тип подъемника'] == 'другое')]

data = data[~(data['Машина'].str.startswith("Телескопический погрузчик"))]

data.loc[data['Машина'] == 'Телескопический подъемник Zoomlion ZT72J-V', 'Тип двигателя'] = 'Дизельный'
data.loc[data['Машина'] == 'Телескопический подъемник Zoomlion ZT30J', 'Тип двигателя'] = 'Дизельный'

data['Дата'] = pd.to_datetime(data['Дата'], format='%d.%m.%Y %H:%M:%S')
data['Год'] = data['Дата'].dt.year
data['Квартал'] = data['Дата'].dt.quarter
data['Лот.CRM год выпуска'] = data['Лот.CRM год выпуска'].apply(lambda x: int(x.replace(',', '')) if isinstance(x, str) else x)
data['Средняя наработка (лет)'] = data['Год'] - data['Лот.CRM год выпуска']

col = data.pop("Квартал")  
data.insert(2, "Квартал", col)

data['Номера машин'] = data.apply(lambda row: extract_model_case_insensitive(row['Машина'], row['Машина.Бренд']), axis=1)

Файл успешно загружен из CSV: Запчасти списанные в ремонт.csv


In [70]:
df1 = copy.deepcopy(data)

In [71]:
df1.loc[df1['Номенклатура.Оригинальный номер расширенный'] == 'JCPT1412HA','Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура.Оригинальный номер расширенный'] == 'AMWP11.5-8100','Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура.Оригинальный номер расширенный'] == 'ZS1212','Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура.Артикул'] == 'DIFA438201', 'Номенклатура.Оригинальный номер расширенный'] = '4382-01'
df1.loc[df1['Номенклатура'] == 'Фильтр воздушный 32925682', 'Номенклатура.Оригинальный номер расширенный'] = '32/925682 AF26656'
df1.loc[df1['Номенклатура'] == 'Фильтр воздушный 32925682', 'Номенклатура.Оригинальный номер'] = '32/925682'
df1.loc[df1['Номенклатура'] == 'НПУ 00007940', 'Номенклатура.Артикул'] = '85000159'
df1.loc[df1['Номенклатура'] == 'НПУ 00007940', 'Номенклатура.Оригинальный номер'] = '85000159'
df1.loc[df1['Номенклатура'] == 'НПУ 00007940', 'Номенклатура.Оригинальный номер расширенный'] = '85000159'
df1.loc[df1['Номенклатура'] == 'Цилиндр поворота платформы 1010201914', 'Номенклатура.Оригинальный номер'] = '1010201914'
df1.loc[df1['Номенклатура'] == 'Трос аварийного опускания Haulotte Compact', 'Номенклатура.Оригинальный номер'] = '2420314420'
df1.loc[df1['Номенклатура'] == 'Трос аварийного опускания Haulotte Compact', 'Номенклатура.Оригинальный номер расширенный'] = '2326009100'
df1.loc[df1['Номенклатура'] == 'Фильтр воздушный 90031461A', 'Номенклатура.Оригинальный номер расширенный'] = '43121'
df1.loc[df1['Номенклатура'] == 'Фильтр масляный C-7926', 'Номенклатура.Оригинальный номер расширенный'] = '1009900560 1009901968 15056-32432 15521-32439 16098-31291 1C01032432 1C020-32430 1C020-32432 1C020-32433 1C020-32434 1C020-32434-P 1C02032434 279809 4042216 582018366 807180 HH1C0-32430 HH1C032430 HLX-6995 LF3376 P550318'
df1.loc[df1['Номенклатура'] == 'Датчик 203150000055', 'Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура'] == 'Фильтр топливный Yanmar DF-5543', 'Номенклатура.Оригинальный номер расширенный'] = '201990003015 60356164 7029012 A977950 MIU802421 SF-52010 SF52010 SK48584 SN25127 XJAU01355 XJAU01515 YM129A0055730'
df1.loc[df1['Номенклатура'] == 'Клапан балансировочный поворотного редуктора корзины 1019901661', 'Номенклатура.Оригинальный номер расширенный'] = '1010201471 1010202355 1010202356 ls2-009-180qmajc'
df1.loc[df1['Номенклатура'] == 'Фильтр гидравлический в сборе 00001871', 'Номенклатура.Оригинальный номер расширенный'] = '00004696 0160D010BN3HC SH75028'
df1.loc[df1['Номенклатура'] == 'Фильтр воздушный A8506S', 'Номенклатура.Оригинальный номер расширенный'] = '1000100310 P827653 RF3091AB ST40706'
df1.loc[df1['Номенклатура'] == 'Фильтр топливный SFC7945002', 'Номенклатура.Оригинальный номер расширенный'] = '1010601542 SFC-79450-02 SFC7945002 SK3170 ST20094'
df1.loc[df1['Номенклатура'] == 'Фильтр гидравлический CS05030P10A', 'Номенклатура.Оригинальный номер расширенный'] = '202080000036 SH56560 SP-06X10G3 SPF-53 SPF-538-100 stx202027'
df1.loc[df1['Номенклатура'] == 'Элемент фильтра гидравлического 00004416', 'Номенклатура.Оригинальный номер расширенный'] = '00003113 00004416'
df1.loc[df1['Номенклатура'] == 'Стартер Kubota 2403 TT15790', 'Номенклатура.Оригинальный номер расширенный'] = '17123-6301-6 17123-63017 1712363016 1712363017 CPK00215'
df1.loc[df1['Номенклатура'] == 'Фильтр топливный FS36209', 'Номенклатура.Оригинальный номер расширенный'] = '1000000876 201990000029'
df1.loc[df1['Номенклатура'] == 'Фильтр топливный ГО BA16-22CRT 10003966A (Kubota)', 'Номенклатура.Оригинальный номер расширенный'] = '1g311-43380 1g31143380 8972133810 FF5468 PF9873 SK3686 SN21586'
df1.loc[df1['Номенклатура'] == 'Фильтр топливный GB6245', 'Номенклатура.Оригинальный номер расширенный'] = '1000700908 PL420 ST21350'
df1.loc[df1['Номенклатура'] == 'Фильтр масляный MANN-W 814/80', 'Номенклатура.Оригинальный номер расширенный'] = '11923630 199100323 2151740 2151740 749613 HH164-32430'
df1.loc[df1['Номенклатура'] == 'Фильтр воздушный DIFA 43121', 'Номенклатура.Оригинальный номер расширенный'] = '90029290 90031461A AF26656 P608533 P609490 ST40110 Z32540'
df1.loc[df1['Номенклатура'] == 'Фильтр гидравлический R0110A10NHA', 'Номенклатура.Оригинальный номер расширенный'] = 'LH0110R010BN/HC DH3723 RHR110G10B3/AB1 P170604'
df1.loc[df1['Номенклатура'] == 'ЭБУ 203020000001', 'Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура'] == 'Зарядное устройство 2029900000311', 'Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура'] == 'Насос гидравлический 81013GT', 'Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура'] == 'Фильтрующий элемент очистки воздуха (внутренний) 4382-01', 'Номенклатура.Оригинальный номер расширенный'] = 'DIFA 4382-01'
df1.loc[df1['Номенклатура'] == 'Фильтрующий элемент очистки воздуха (внутренний) 4382-01', 'Номенклатура.Артикул'] = 'DIFA 4382-01'
df1.loc[df1['Номенклатура'] == 'Замок зажигания 90173', 'Номенклатура.Оригинальный номер расширенный'] = '84830'
df1.loc[df1['Номенклатура'] == 'Стартер 1002415445С', 'Номенклатура.Оригинальный номер расширенный'] = '1003692596B 1009806744'
# suspecious
df1.loc[df1['Номенклатура'] == 'Крышка гидравлического бака (ВА 16-22)', 'Номенклатура.Оригинальный номер расширенный'] = '90003907 90003907A'
df1.loc[df1['Номенклатура'] == 'Двигатель вращения 00004901', 'Номенклатура.Оригинальный номер расширенный'] = '0000490'
df1.loc[df1['Номенклатура'] == 'Цилиндр выравнивания платформы 77370030100000', 'Номенклатура.Оригинальный номер расширенный'] = '773700301000001'
df1.loc[df1['Номенклатура'] == 'Кронштейн крепления гидравлического фильтра 104024023003B', 'Номенклатура.Оригинальный номер расширенный'] = '104024023003B02'
df1.loc[df1['Номенклатура'] == 'Замок зажигания 4130000723', 'Номенклатура.Оригинальный номер расширенный'] = '4130000723001'
df1.loc[df1['Номенклатура'] == 'фильтр воздушный! D105.6 d60 H274 \ Caterpillar, Bobcat, Komatsu, Kubota', 'Номенклатура.Оригинальный номер расширенный'] = '91745'
df1.loc[df1['Номенклатура'] == 'Контактор 24V 200A  28.01.000285', 'Номенклатура.Оригинальный номер расширенный'] = '2801000285'
# 
# df1.loc[df1['Номенклатура'] == 'Опора аутригера 10003999A (82мм)', 'Номенклатура.Оригинальный номер расширенный'] = '10003999'
# df1.loc[df1['Номенклатура'] == 'Подшипник 33109A ZKL', 'Номенклатура.Оригинальный номер расширенный'] = '33109'
# df1.loc[df1['Номенклатура'] == 'Ремень клиновой AVX10X1125', 'Номенклатура.Оригинальный номер расширенный'] = '10X1125'
# df1.loc[df1['Номенклатура'] == 'Кабель рабочей платформы папа 1202000009', 'Номенклатура.Оригинальный номер расширенный'] = '12.02.000009 1202000009'
# df1.loc[df1['Номенклатура'] == 'Пол платформы BA24-28 92100316A', 'Номенклатура.Оригинальный номер расширенный'] = '92100316'
# df1.loc[df1['Номенклатура'] == 'Подшипник 4Т-33006 NTN', 'Номенклатура.Оригинальный номер расширенный'] = '33006'
# df1.loc[df1['Номенклатура'] == 'Клапан (правый) наклона оси в сборе с датчиком 92047789', 'Номенклатура.Оригинальный номер расширенный'] = '92047789A'
# df1.loc[df1['Номенклатура'] == 'Соленоид электромагнитного клапана поворота колёс 92047788', 'Номенклатура.Оригинальный номер расширенный'] = '92047788A'
# df1.loc[df1['Номенклатура'] == 'Насос 00007996A', 'Номенклатура.Оригинальный номер расширенный'] = '00000529 00012585 00012585A'
# df1.loc[df1['Номенклатура'] == 'Крепеж опоры 10004004A', 'Номенклатура.Оригинальный номер расширенный'] = '10004004'
# df1.loc[df1['Номенклатура'] == 'Зарядное устройство Dingli BA18NE  00007381', 'Номенклатура.Оригинальный номер расширенный'] = '00007381A'
# df1.loc[df1['Номенклатура'] == 'Клапан гидрораспределителя 00011000', 'Номенклатура.Оригинальный номер расширенный'] = '00011000A'
# df1.loc[df1['Номенклатура'] == 'Маячок 00005190A', 'Номенклатура.Оригинальный номер расширенный'] = '00005190'
# df1.loc[df1['Номенклатура'] == 'НПУ 00010950A', 'Номенклатура.Оригинальный номер расширенный'] = '00010950'
#
df1.loc[df1['Номенклатура'] == 'Фильтр гидравлический RHR110G10B3/AB1', 'Номенклатура.Оригинальный номер расширенный'] = '(LH)0110A10BN/HA(HC) 0110R010BN3HC KFSF-0110-R-010-N R0110 R0110A010NHA R0110A10NHA'

df1.loc[df1['Номенклатура'] == 'Фильтр воздушный Z32540', 'Номенклатура.Оригинальный номер'] = '90029290'

df1.loc[df1['Номенклатура'] == 'Фильтрующий элемент очистки воздуха (внешний) 4382', 'Номенклатура.Оригинальный номер расширенный'] = np.nan

df1.loc[df1['Номенклатура'] == 'Фильтр воздушный внутренний P829333', 'Номенклатура.Оригинальный номер расширенный'] = '216070000008 4383-01'

df1.loc[df1['Номенклатура'] == 'Фильтр воздушный внешний P828889', 'Номенклатура.Оригинальный номер расширенный'] = '216070000004 4383'

df1.loc[df1['Номенклатура'] == 'Фильтр масляный C1110', 'Номенклатура.Оригинальный номер расширенный'] = '90024289 SO228 C-1405 LS32665 01174416'

df1.loc[df1['Номенклатура'] == 'Фильтр воздушный комплект p828889', 'Номенклатура.Оригинальный номер расширенный'] = '216070000004 4383'
df1.loc[df1['Номенклатура'] == 'Фильтр воздушный комплект p828889', 'Номенклатура.Оригинальный номер'] = '00006753'

df1.loc[df1['Номенклатура'] == 'Фильтр гидравлический D0110A10NHA', 'Номенклатура.Оригинальный номер расширенный'] = '00004697 D0110A10NHA LH0110D010BN3HC'

df1.loc[df1['Номенклатура'] == 'Фильтр воздушный DIFA 43121-01', 'Номенклатура.Артикул'] = 'DIFA 43121-01'

df1.loc[df1['Номенклатура'] == 'Фара 00006753A/00005190A', 'Номенклатура.Артикул'] = '00006753'
df1.loc[df1['Номенклатура'] == 'Фара 00006753A/00005190A', 'Номенклатура'] = 'Фара головного света 00006753'
df1.loc[df1['Номенклатура'] == 'Фильтр масляный C-1049', 'Номенклатура.Артикул'] = 'C1049'

cols = [
    'Номенклатура.Артикул',
    'Номенклатура.Оригинальный номер',
    'Номенклатура.Оригинальный номер расширенный'
]

df1[cols] = df1[cols].replace(r'DIFA\s*', '', regex=True)

df1[cols] = df1[cols].replace(r'\s+', ' ', regex=True).apply(lambda x: x.str.strip())

df1.loc[df1['Номенклатура.Артикул'] == '438201', 'Номенклатура.Артикул'] = '4382-01'

mask_plus = (
    df1['Номенклатура'].str.contains(r'\+', na=False) &
    ~df1['Номенклатура'].str.contains(r'^(?:Колесо|РВД|[Оо]богреватель|Контроллер|Deutz|Выключатель|Батарея|Рукав|Фильтр воздушный к-кт \(внутр\.\+внешн\.\) 1351230502)', na=False) # Учесть фильтр последний (т.е найти его аналоги)
)

mask_st_units = df1['Номенклатура'].str.contains(
    r'Фильтр воздушный ST40111/40110',
    case=False,
    na=False
)

mask_brackets = df1['Номенклатура'].str.contains(
    r'(95*165*340/70*90*335,',
    case=False,
    na=False,
    regex=False
)

mask_kt = df1['Номенклатура'].str.contains(
    r'Фильтр воздушный AF26656 к-т',
    case=False,
    na=False,
)

mask_AB = df1['Номенклатура'].str.contains(
    r'Фильтр воздушный ST40131AB',
    case=False,
    na=False,
)

mask_hid = df1['Номенклатура'].str.contains(
    r'Фильтр воздушный ST40111ST40110',
    case=False,
    na=False,
)

complects = df1[mask_plus | mask_st_units | mask_brackets | mask_kt | mask_AB | mask_hid]
complects.loc[complects['Номенклатура'] == 'Фильтр воздушный AF26656 к-т', 'Номенклатура'] = 'Фильтр воздушный к-т AF26656+AF26655'
complects.loc[complects['Номенклатура'] == 'Фильтр воздушный ST40131AB', 'Номенклатура'] = 'Фильтр воздушный (95*165*340/70*90*335, сдвоенный) kw1634'

df1 = df1[~df1.index.isin(complects.index)]


In [72]:
matches = {
    '43121': '90029290 90031461A AF26656 P608533 P609490 ST40110 Z32540', # внешниий
    '43121-01': '90031459 P600975 Z32603', # внутренний
    '4382': '1000100310 P827653', # внешниий
    '4382-01': '1000100311 P829332', # внутренний
    'kw1634': '201020003059', # внешниий
    'kw1634in': '201020003058', # внутренний
    'AF26656': '32/925682 ST40110', # внешниий
    'AF26655': '32/925683 ST40111', # внутренний
    'ST40110': '90031461 AF26656', # внешний
    'ST40111': '90031459 AF26655', # внутренний
    '4383': '216070000004 P828889', # внешниий
    '4383-01': '216070000008 P829333', # внутренний
    'AF2555': '1000100310 6666375 B222100000593', # внешниий
    'AF25484': '1000100311 6666376 B222100000591', # внутренний
    'AP31642K': '216070000004', # внешний
    'AP31643K': '216070000008', # внутренний. Возможно аналог DIFA 4383-01
    '6666375': '1000100310', # внешниий
    '6666376': '1000100311', # внутренний
    '1000100310': '4382', # внешний
    '1000100311': '4382-01', # внутренний
    '43123': '43123', # внешний
    '43123-01': '43123-01', # внутренний
    '4338': '4338', # внешний
    '4338-01': '4338-01', # внутренний
    'ST40722(1)': '4338',  # внутренний
    'ST40722(2)': '4338-01'  # внутренний
}
	

complects['Номенклатура.Артикул'] = complects['Номенклатура'].apply(lambda x: extract_articles(x, matches.keys()))

df_exploded = complects.explode('Номенклатура.Артикул').reset_index(drop=True)

df_exploded['Номенклатура.Оригинальный номер'] = df_exploded['Номенклатура.Артикул'].map(lambda x: matches[x].split()[0] if x in matches else None)
df_exploded['Номенклатура.Оригинальный номер расширенный'] = df_exploded['Номенклатура.Артикул'].map(matches)

assert (df_exploded.shape[0] == complects.shape[0] * 2)

df1 = pd.concat([df1, df_exploded], ignore_index=True)

df1 = df1.sort_values(by='Дата', ignore_index=True)

In [73]:
cols = [
    'Номенклатура.Артикул',
    'Номенклатура.Оригинальный номер',
    'Номенклатура.Оригинальный номер расширенный'
]

for i in cols:
    df1[i] = df1[i].apply(normalize)

df1["Номенклатура.Оригинальный номер расширенный"] = df1.apply(add_original_to_extended, axis=1) 

df1['Аналоги'] = (
    df1['Номенклатура.Оригинальный номер расширенный']
    .fillna('')         
    .str.upper()
    .str.split()
)

graph = defaultdict(set)

for _, row in df1.iterrows():
    # все формы основного артикула (с суффиксом и без)
    part_forms = article_forms(row['Номенклатура.Артикул'])

    # связать формы одного артикула между собой
    for i in range(len(part_forms)):
        for j in range(i + 1, len(part_forms)):
            graph[part_forms[i]].add(part_forms[j])
            graph[part_forms[j]].add(part_forms[i])

    # связать с аналогами
    for analog_raw in row['Аналоги']:
        for af in article_forms(analog_raw):
            for pf in part_forms:
                if pf != af:
                    graph[pf].add(af)
                    graph[af].add(pf)


df1['all_analogs'] = df1['Номенклатура.Артикул'].apply(lambda x: find_all_analogs(x, graph))
df1 = df1.drop(columns='Аналоги')

group_mapping = {}
group_counter = 1

for group_tuple in df1['all_analogs'].unique():
    group_mapping[group_tuple] = group_counter
    group_counter += 1

df1['Номер группы'] = df1['all_analogs'].map(group_mapping)
df1.to_excel('text.xlsx', index=False)
df_quarter = merge_parts_df1(df1)

In [74]:
data2 = load_repair_parts('Остатки и обороты.csv', 'Остатки и обороты.xlsx', skiprows=8)
data2 = data2.iloc[:-2].reset_index(drop=True)
data2 = data2.rename(columns={'Артикул ': 'Артикул'})

period_mask = data2['Номенклатура'].str.contains(
    r'\d{4} г\.|\d квартал \d{4} г\.|Январь|Февраль|Март|Апрель|Май|Июнь|Июль|Август|Сентябрь|Октябрь|Ноябрь|Декабрь',
    na=False
)

data2['year'] = data2['Номенклатура'].str.extract(r'\b(20[2-9]\d)\b')
data2['quarter'] = data2['Номенклатура'].str.extract(r'(\d квартал)')
data2['month'] = data2['Номенклатура'].str.extract(
    r'(Январь|Февраль|Март|Апрель|Май|Июнь|Июль|Август|Сентябрь|Октябрь|Ноябрь|Декабрь)'
)
data2[['year','quarter','month']] = data2[['year','quarter','month']].ffill()
data2 = data2[~period_mask]

data2['Квартал'] = data2['quarter'].str.extract(r'(\d)').astype(int)

month_map = {
    'Январь': 1, 'Февраль': 2, 'Март': 3, 'Апрель': 4, 'Май': 5, 'Июнь': 6,
    'Июль': 7, 'Август': 8, 'Сентябрь': 9, 'Октябрь': 10, 'Ноябрь': 11, 'Декабрь': 12
}
data2['Месяц'] = data2['month'].map(month_map).astype(int)
data2['Год'] = data2['year']
data2.drop(columns=['Unnamed: 0', 'month', 'quarter', 'year'], inplace=True)
data2 = data2.loc[(data2['Артикул'].notna()) | (data2['Оригинальный номер'].notna())]

for col in ["Расход", "Приход", "Конечный остаток"]:
    data2[col] = (
        data2[col]
        .str.replace(',', '', regex=False)
        .astype(float)
    )
# data2.to_excel('test2.xlsx', index=False)

Файл успешно загружен из CSV: Остатки и обороты.csv


In [75]:
df2 = copy.deepcopy(data2)

In [76]:
cols = [
    'Артикул',
    'Оригинальный номер',
]

df2[cols] = df2[cols].replace(r'DIFA\s*', '', regex=True)

df2[cols] = df2[cols].replace(r'\s+', ' ', regex=True).apply(lambda x: x.str.strip())

df2.loc[df2['Артикул'] == '32925682', 'Оригинальный номер'] = '32925682'
df2.loc[df2['Артикул'] == '4312101', 'Артикул'] = '43121-01'
df2.loc[df2['Артикул'] == '438201', 'Артикул'] = '4382-01'
df2.loc[df2['Артикул'] == '438301', 'Артикул'] = '4383-01'
df2.loc[df2['Номенклатура'] == 'НПУ 00007940', 'Артикул'] = '85000159'
df2.loc[df2['Номенклатура'] == 'Насос подкачки 4128101', 'Артикул'] = '04128101'
df2.loc[df2['Номенклатура'] == 'Фильтр масляный C-1049', 'Артикул'] = 'C1049'
df2.loc[df2['Номенклатура'] == 'Насос подкачки 4128101', 'Оригинальный номер'] = '04128101'
df2.loc[df2['Номенклатура'] == 'Цепь ГРМ Cummins ISF2.8', 'Оригинальный номер'] = '4982040f'
df2.loc[df2['Номенклатура'] == 'Насос 00003102A', 'Оригинальный номер'] = '00003102'
df2.loc[df2['Номенклатура'] == 'Колесо в сборе 105051034051A', 'Артикул'] = '105051034051A'
df2.loc[df2['Номенклатура'] == 'Предохранитель автоматический 30А  291.3722,', 'Артикул'] = '291.3722'
df2.loc[df2['Номенклатура'] == 'Предохранитель автоматический 30А  291.3722,', 'Оригинальный номер'] = '291.3722'

mask_plus = (
    df2['Номенклатура'].str.contains(r'\+', na=False) &
    ~df2['Номенклатура'].str.contains(
        r'^(Колесо|РВД|[Оо]богреватель|Распределитель|Насос|Комплект|Кабель|Гидроцилиндр|Датчик|Коллектор|Фильтр топливный PERKINS|Фильтр воздушный \(внешний\+внутренний\) A5541S)',
        na=False
    )
)

mask_st_units = df2['Номенклатура'].str.contains(
    r'Фильтр воздушный ST40111/40110',
    case=False,
    na=False
)
mask_brackets = df2['Номенклатура'].str.contains(
    r'(95*165*340/70*90*335,',
    case=False,
    na=False,
    regex=False
)

mask_kt = df2['Номенклатура'].str.contains(
    r'Фильтр воздушный AF26656 к-т',
    case=False,
    na=False,
)

mask_AB = df2['Номенклатура'].str.contains(
    r'Фильтр воздушный ST40131AB',
    case=False,
    na=False,
)

mask_hid = df2['Номенклатура'].str.contains(
    r'Фильтр воздушный ST40111ST40110',
    case=False,
    na=False,
)

mask_filter = df2['Номенклатура'].str.contains(
    r'Фильтр воздушный 6666375/6666376',
    case=False,
    na=False,
)

complects2 = df2[mask_plus | mask_st_units | mask_brackets | mask_kt | mask_AB | mask_hid | mask_filter]
complects2.loc[complects2['Номенклатура'] == 'Фильтр воздушный AF26656 к-т', 'Номенклатура'] = 'Фильтр воздушный к-т AF26656+AF26655'
complects2.loc[complects2['Номенклатура'] == 'Фильтр воздушный ST40131AB', 'Номенклатура'] = 'Фильтр воздушный (95*165*340/70*90*335, сдвоенный) kw1634'

df2 = df2[~df2.index.isin(complects2.index)]

In [77]:
complects2['Артикул'] = complects2['Номенклатура'].apply(lambda x: extract_articles(x, matches.keys()))

df_exploded = complects2.explode('Артикул').reset_index(drop=True)

df_exploded['Оригинальный номер'] = df_exploded['Артикул'].map(lambda x: matches[x].split()[0] if x in matches else None)

df_exploded[df_exploded['Артикул'].isna()]

assert (df_exploded.shape[0] == complects2.shape[0] * 2)

df2 = pd.concat([df2, df_exploded], ignore_index=True)

In [78]:
df2["Артикул"]  = df2["Артикул"].apply(normalize)
df2["Оригинальный номер"] = df2["Оригинальный номер"].apply(normalize)

for i in cols:
    df2[i] = df2[i].str.strip().str.upper()

def lookup(art_norm, orig_norm):
    for val in [art_norm, orig_norm]:
        if not val:
            continue
        for form in article_forms(val):
            if form in article_to_group:
                return article_to_group[form], article_to_analogs[form]
    return None, None

article_to_group   = {}
article_to_analogs = {}

for _, row in df1.drop_duplicates('Номенклатура.Артикул').iterrows():
    group = row['Номер группы']
    raw = row['all_analogs']
    try:
        analogs = raw if isinstance(raw, tuple) else ast.literal_eval(raw)
    except:
        analogs = ()
    for art in analogs:
        article_to_group[art]   = group
        article_to_analogs[art] = analogs
    main = row['Номенклатура.Артикул']
    if pd.notna(main) and main:
        article_to_group[main]   = group
        article_to_analogs[main] = analogs


groups, analogs_list = zip(
    *df2.apply(lambda r: lookup(r["Артикул"], r["Оригинальный номер"]), axis=1)
)

df2["Номер группы"] = groups
df2["Список аналогов"] = analogs_list

def enrich_analogs(row):
    art = row["Артикул"]
    analogs = row["Список аналогов"]
    if pd.isnull(art) or not isinstance(analogs, tuple):
        return analogs
    if art not in analogs and art not in article_to_group:
        return tuple(sorted(analogs + (art,)))
    return analogs

df2["Список аналогов"] = df2.apply(enrich_analogs, axis=1)
df2 = normalize_analog_lists(df2)

In [79]:
graph_new = defaultdict(set)

for idx in df2[df2["Номер группы"].isna()].index:
    art = normalize(df2.at[idx, "Артикул"])
    orig = normalize(df2.at[idx, "Оригинальный номер"])

    if art is None:
        if orig is not None:
            art = orig
        else:
            continue

    # проверяем все формы, не только точную
    all_forms = article_forms(art) + (article_forms(orig) if orig else [])
    if any(f in article_to_group for f in all_forms):
        continue

    for af in article_forms(art):
        for bf in article_forms(art):
            if af != bf:
                graph_new[af].add(bf)
                graph_new[bf].add(af)

    if orig is not None:
        for af in article_forms(art):
            for bf in article_forms(orig):
                if af != bf:
                    graph_new[af].add(bf)
                    graph_new[bf].add(af)


art_by_group = (
    df2[df2["Номер группы"].notna()]
    .assign(Артикул_norm=lambda d: d["Артикул"].apply(normalize))
    .dropna(subset=["Артикул_norm"])
    .groupby(df2["Номер группы"].dropna().astype(str))["Артикул_norm"]
    .apply(set)
    .to_dict()
)

df1_group_idx = defaultdict(list)
for i, g in df1["Номер группы"].astype(str).items():
    df1_group_idx[g].append(i)

df2_group_idx = defaultdict(list)
for i, g in df2["Номер группы"].dropna().astype(str).items():
    df2_group_idx[g].append(i)

for grp, arts in art_by_group.items():
    idxs_df1 = df1_group_idx.get(grp, [])
    if not idxs_df1:
        continue
    existing = df1.at[idxs_df1[0], "all_analogs"]
    if not isinstance(existing, tuple):
        continue
    new_arts = arts - set(existing)
    if not new_arts:
        continue
    new_analogs = tuple(sorted(existing + tuple(new_arts)))
    for i in idxs_df1:
        df1.at[i, "all_analogs"] = new_analogs
    for i in df2_group_idx.get(grp, []):
        df2.at[i, "Список аналогов"] = new_analogs


for idx in df2[df2["Номер группы"].isna()].index:
    art = normalize(df2.at[idx, "Артикул"])
    orig = normalize(df2.at[idx, "Оригинальный номер"])

    if art is None:
        art = orig

    all_forms = article_forms(art) if art else []
    if any(f in article_to_group for f in all_forms):
        continue

    if art is not None:
        df2.at[idx, "Список аналогов"] = find_all_analogs(art, graph_new)
    else:
        df2.at[idx, "Список аналогов"] = tuple()


unique_new_analogs = df2.loc[df2["Номер группы"].isna(), "Список аналогов"].drop_duplicates()
new_group = df1["Номер группы"].max() + 1
group_mapping = {grp: new_group + i for i, grp in enumerate(unique_new_analogs)}

mask = df2["Номер группы"].isna()
df2.loc[mask, "Номер группы"] = df2.loc[mask, "Список аналогов"].apply(
    lambda x: group_mapping.get(x)
)

In [ ]:
# # suspecious
# mixed = find_mixed_groups(df2)

# suspicious = mixed[mixed['схожесть_названий'] <= 1]

# result = (
#     df1[df1['Номер группы'].isin(suspicious['Номер группы'])]
#     [['Номер группы', 'Номенклатура.Артикул', 'Номенклатура']]
#     .drop_duplicates()
#     .sort_values('Номер группы')
# )

# result.to_excel("mixed_groups.xlsx", index=False)


# group_ids = result['Номер группы'].ne(result['Номер группы'].shift()).cumsum()

# palette = ["#FFCCCC","#CCFFCC","#CCCCFF","#FFFFCC","#CCFFFF"]

# colors = {g: palette[i % len(palette)] for i, g in enumerate(group_ids.unique())}

# def color_rows(row):
#     color = colors[group_ids.loc[row.name]]
#     return [f'background-color: {color}'] * len(row)

# styled = result.style.apply(color_rows, axis=1)

# styled.to_excel("colored.xlsx", engine="openpyxl", index=False)

In [81]:
agg = merge_parts_df2(df2)
agg = prepare_sales(agg, df_quarter)

In [82]:
final_df = pd.merge(
    df_quarter, 
    agg, 
    on=['Год','Месяц','Номер группы'], 
    how='outer', 
    suffixes=('_df','_agg'),
)

cols_to_combine = [c for c in df_quarter.columns if c in agg.columns and c not in ['Год','Месяц','Номер группы']]

for col in cols_to_combine:
    final_df[col] = final_df[f'{col}_df'].combine_first(final_df[f'{col}_agg'])

cols_to_drop = [c for c in final_df.columns if c.endswith('_df') or c.endswith('_agg')]

final_df.drop(columns=cols_to_drop, inplace=True)

final_df['Продажа'] = final_df['Продажа'].fillna(0)
final_df['Ремонт'] = final_df['Ремонт'].fillna(0)

cols = [c for c in final_df.columns if c not in ['Продажа', 'Ремонт', 'Приход', 'Конечный остаток']]

final_df = final_df[cols + ['Приход', 'Конечный остаток', 'Продажа', 'Ремонт']]

final_df = normalize_analog_lists(
    final_df,
    col_group="Номер группы",
    col_analogs="Список аналогов"
)

name_map = (
    final_df.groupby('Номер группы')['Номенклатура']
    .apply(lambda s: min(s.dropna().astype(str), key=len, default=None))
)
final_df['Номенклатура'] = final_df['Номер группы'].map(name_map)

In [ ]:
# all_groups = []
# seen = set()

# for node in graph:
#     if node not in seen:
#         group_nodes = set(find_all_analogs(node, graph))  # преобразуем tuple в set
#         all_groups.append(group_nodes)
#         seen |= group_nodes

# all_groups.sort(key=len, reverse=True)

# top_n = 5
# top_groups = all_groups[:top_n]

# for i, group in enumerate(top_groups, 1):
#     G = nx.Graph()
#     for node in group:
#         for neighbor in graph[node]:
#             if neighbor in group:
#                 G.add_edge(node, neighbor)
    
#     plt.figure(figsize=(16,10))
#     pos = nx.spring_layout(G)
#     nx.draw(G, pos, with_labels=True, node_size=2000, font_size=10)
#     plt.title(f"Группа {i} — размер {len(group)}")
#     plt.show()

In [84]:
final_df.to_excel('final.xlsx',index=False)

In [ ]:
# recl = pd.read_excel('Аналитика по рекламации.xlsx', dtype=str)

# mapping = str.maketrans({
#     'А':'A','В':'B','Е':'E','К':'K','М':'M',
#     'Н':'H','О':'O','Р':'P','С':'C','Т':'T','Х':'X'
# })

# # 1. Нормализуем df1
# df1['M_norm'] = (
#     df1['Машина']
#     .astype(str)
#     .str.upper()
#     .str.translate(mapping)
#     .str.replace(r"[^A-Z0-9]", "", regex=True)
# )

# # 2. Нормализуем recl
# recl['Recl_norm'] = (
#     recl['Модель']
#     .astype(str)
#     .str.upper()
#     .str.translate(mapping)
#     .str.replace(r"[^A-Z0-9]", "", regex=True)
# )

# # 3. Создаем список нормализованных машин + каталожный номер
# recl_list = recl[['Recl_norm', 'Каталожный номер ЗЧ']].values.tolist()

# # 4. Функция для поиска совпадений
# def get_recl_cat_nums(m_norm):
#     matches = [cat for recl_val, cat in recl_list if recl_val in m_norm]
#     return ",".join(matches)  # объединяем через запятую

# # 5. Применяем к df1
# df1['recl_cat_nums'] = df1['M_norm'].apply(get_recl_cat_nums)

In [ ]:
# result = audit_analog_groups(final_df)
# export_to_excel(result, "audit_report.xlsx")

✅ Отчёт сохранён: audit_report.xlsx  (28 проблем, 4 листов)


In [ ]:
import pandas as pd
import numpy as np
import ast
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
import os
import sys

warnings.filterwarnings('ignore')

DATA_FILE = "final.xlsx"

# Прогноз на Дек 2025, Янв 2026, Фев 2026
FORECAST_MONTHS = [(2025, 12), (2026, 1), (2026, 2)]
FORECAST_LABELS = ["Дек 2025", "Янв 2026", "Фев 2026"]

# Обучение: всё до декабря 2025
TRAIN_END_YEAR, TRAIN_END_MONTH = 2025, 11


# ── Загрузка ──────────────────────────────────────────────────────────────────
def load_data(filepath):
    df = pd.read_excel(filepath)
    df['all_analogs'] = df['Список аналогов']
    df.drop(columns='Список аналогов', inplace=True)
    df_train = df[
        ~((df['Год'] == 2025) & (df['Месяц'] == 12)) &
        ~(df['Год'] == 2026)
    ].copy()
    df_train['date'] = pd.to_datetime(
        df_train['Год'].astype(str) + '-' +
        df_train['Месяц'].astype(str).str.zfill(2) + '-01'
    )
    monthly = (df_train
               .groupby(['date', 'Номер группы'])[['Продажа', 'Ремонт']]
               .sum()
               .reset_index())

    # Факт за Дек 25 / Янв 26 / Фев 26
    actual = {}
    for y, m in FORECAST_MONTHS:
        sub = df[(df['Год'] == y) & (df['Месяц'] == m)]
        for _, row in sub.iterrows():
            g = int(row['Номер группы']) if not pd.isna(row['Номер группы']) else None
            if g is None:
                continue
            if g not in actual:
                actual[g] = {'sale': [0, 0, 0], 'repair': [0, 0, 0]}
            idx = FORECAST_MONTHS.index((y, m))
            actual[g]['sale'][idx]   += float(row['Продажа'] or 0)
            actual[g]['repair'][idx] += float(row['Ремонт']  or 0)

    # Мета
    def parse_analogs(v):
        if pd.isna(v) or str(v).strip() in ('', 'nan'):
            return ''
        try:
            return ', '.join(str(x) for x in ast.literal_eval(str(v)))
        except Exception:
            return str(v).strip("()'\"")

    group_meta = (df.groupby('Номер группы')
                  .agg(Номенклатура=('Номенклатура', 'first'),
                       Артикул=('Артикул', 'first'),
                       ТипТехники=('Тип техники', 'first'),
                       all_analogs=('all_analogs', 'first'))
                  .reset_index())
    group_meta['Аналоги'] = group_meta['all_analogs'].apply(parse_analogs)
    group_meta['Номер группы'] = group_meta['Номер группы'].astype(int)

    all_groups = set(group_meta['Номер группы'].tolist())
    return df_train, monthly, group_meta, actual, all_groups


# ── Обнаружение и замена выбросов ─────────────────────────────────────────────
def remove_outliers(series: pd.Series, iqr_factor: float = 1.5) -> tuple:
    """
    Заменяет выбросы (> Q3 + factor*IQR) на скользящую медиану по 3 соседям.
    Возвращает (очищенный ряд, список выбросов [(дата, исходное, замена)]).
    """
    s = series.copy()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    upper = q3 + iqr_factor * iqr

    # Нижний порог не используем (нули — норма для запчастей)
    outlier_mask = s > upper

    outlier_log = []
    for idx in s[outlier_mask].index:
        original = s[idx]
        # Медиана ближайших 3 соседей (без самого выброса)
        pos = s.index.get_loc(idx)
        neighbors = []
        for offset in [-3, -2, -1, 1, 2, 3]:
            ni = pos + offset
            if 0 <= ni < len(s) and not outlier_mask.iloc[ni]:
                neighbors.append(s.iloc[ni])
        replacement = float(np.median(neighbors)) if neighbors else float(s.median())
        s.iloc[pos] = replacement
        outlier_log.append({
            'date':        idx.strftime('%Y-%m'),
            'original':    round(original, 1),
            'replacement': round(replacement, 1),
            'threshold':   round(upper, 1),
        })

    return s, outlier_log


# ── Прогнозирование ─────────────────────────────────────────────────────────
ZERO_THRESHOLD = 0.40   # доля нулей > 40% → Croston

def get_series(monthly, grp, col):
    full_dates = pd.date_range('2023-01-01', '2025-11-01', freq='MS')
    sub = (monthly[monthly['Номер группы'] == grp][['date', col]]
           .set_index('date'))
    return sub[col].reindex(full_dates, fill_value=0).astype(float)


def forecast_croston(series, steps=3):

    s = series.clip(lower=0).values
    alpha = 0.1
    nonzero = [(i, v) for i, v in enumerate(s) if v > 0]
    if not nonzero:
        return pd.Series([0.0] * steps,
                         index=pd.date_range('2025-12-01', periods=steps, freq='MS')), 'Croston'
    z = nonzero[0][1]
    p = float(nonzero[0][0] + 1)
    prev_idx = nonzero[0][0]
    for idx, val in nonzero[1:]:
        interval = idx - prev_idx
        z = alpha * val      + (1 - alpha) * z
        p = alpha * interval + (1 - alpha) * p
        prev_idx = idx
    fc_val = round(z / p, 1) if p > 0 else 0.0
    fc = pd.Series([max(0.0, fc_val)] * steps,
                   index=pd.date_range('2025-12-01', periods=steps, freq='MS'))
    return fc, 'Croston'


def forecast_auto_ets(series, steps=3):

    s = series.clip(lower=0)
    has_zeros = (s == 0).any()
    candidates = [
        ('add', 'add',  12, 'HW-add'),
        ('add', 'mul',  12, 'HW-mul'),
        ('add', None,   None, 'Holt (тренд)'),
        (None,  'add',  12,  'ETS (сезон.)'),
        (None,  None,   None, 'Simple ETS'),
        ('add', 'add',  6,   'HW-add (6мес)'),
    ]
    best_fc, best_aic, best_name = None, np.inf, None
    for trend, seasonal, sp, name in candidates:
        if seasonal == 'mul' and has_zeros:
            continue
        if sp and len(s) < 2 * sp:
            continue
        try:
            m = ExponentialSmoothing(s, trend=trend, seasonal=seasonal,
                                     seasonal_periods=sp)
            f = m.fit(optimized=True)
            if f.aic < best_aic:
                best_aic  = f.aic
                best_fc   = f.forecast(steps).clip(lower=0).round(1)
                best_name = name
        except Exception:
            pass
    if best_fc is None:
        v = s.tail(6).mean()
        best_fc = pd.Series([round(v, 1)] * steps,
                            index=pd.date_range('2025-12-01', periods=steps, freq='MS'))
        best_name = 'Mean-6m'
    return best_fc, best_name


def forecast_best(series, steps=3):
    s = series.clip(lower=0)
    zero_ratio = (s == 0).sum() / len(s)
    if zero_ratio > ZERO_THRESHOLD:
        return forecast_croston(s, steps)
    return forecast_auto_ets(s, steps)


def forecast_group(monthly, group_meta, actual, grp):
    meta_row = group_meta[group_meta['Номер группы'] == grp]
    if meta_row.empty:
        return None
    meta = meta_row.iloc[0]
    result = {'grp': grp,
              'name':    str(meta['Номенклатура'])[:70],
              'article': str(meta['Артикул']),
              'type':    str(meta['ТипТехники']),
              'analogs': str(meta['Аналоги']),
              'sale':   {},
              'repair': {}}
    for col, key in [('Продажа', 'sale'), ('Ремонт', 'repair')]:
        raw = get_series(monthly, grp, col)
        zero_ratio = int(round((raw == 0).sum() / len(raw) * 100, 0))
        fc_raw,   method_raw   = forecast_best(raw, 3)
        cleaned, outlier_log   = remove_outliers(raw)
        fc_clean, method_clean = forecast_best(cleaned, 3)
        act_vals = actual.get(grp, {}).get(key, [0, 0, 0])
        result[key] = {
            'fc_raw':       [round(float(fc_raw.iloc[i]),   1) for i in range(3)],
            'fc_clean':     [round(float(fc_clean.iloc[i]), 1) for i in range(3)],
            'act':          [round(float(v), 1) for v in act_vals],
            'method_raw':   method_raw,
            'method_clean': method_clean,
            'outliers':     outlier_log,
            'zero_ratio':   zero_ratio,
        }
    return result

def build_forecast_rows(groups, monthly, group_meta, actual, col_key):
    rows = []
    for grp in groups:
        print(f"    → Группа {grp}...", end='', flush=True)
        r = forecast_group(monthly, group_meta, actual, grp)
        if r:
            rows.append(r)
            d = r[col_key]
            n_out = len(d['outliers'])
            print(f"  {n_out} выбросов | "
                  f"fc_raw={d['fc_raw']} | fc_clean={d['fc_clean']} | act={d['act']}")
        else:
            print(" — нет данных")
    return rows


C_DARK = "1A2744"; C_MID = "2D4A8A"; C_ACC = "4A90D9"
C_ORG  = "B8520A"; C_ORL = "FFF3CD"
C_GRN  = "1E6E42"; C_GNL = "E9F7EF"
C_RED  = "922B21"; C_RDL = "FDEDEC"
C_STR  = "EBF3FB"; C_WHT = "FFFFFF"
C_TL   = "117A65"; C_TLL = "D5F5E3"
C_PRP  = "5B2C6F"; C_PRL = "F4ECF7"  # purple for cleaned forecast

def fl(c):  return PatternFill("solid", fgColor=c)
def fn(bold=False, color="000000", size=9, italic=False):
    return Font(name='Arial', bold=bold, color=color, size=size, italic=italic)
_t  = Side(border_style="thin",   color="CCCCCC")
_tb = Side(border_style="medium", color="2D4A8A")
BRD = Border(left=_t, right=_t, top=_t, bottom=_t)
CTR = Alignment(horizontal='center', vertical='center', wrap_text=True)
LFT = Alignment(horizontal='left',   vertical='center', wrap_text=True)
RGT = Alignment(horizontal='right',  vertical='center')

def sc(ws, r, c, v, _fn=None, _fl=None, _al=None, fmt=None):
    cell = ws.cell(row=r, column=c, value=v)
    if _fn: cell.font = _fn
    if _fl: cell.fill = _fl
    if _al: cell.alignment = _al
    if fmt: cell.number_format = fmt
    cell.border = BRD

def mc(ws, r1, c1, r2, c2, v, _fn=None, _fl=None, _al=None):
    ws.merge_cells(start_row=r1, start_column=c1, end_row=r2, end_column=c2)
    cell = ws.cell(row=r1, column=c1, value=v)
    if _fn: cell.font = _fn
    if _fl:
        for row in ws.iter_rows(min_row=r1, max_row=r2, min_col=c1, max_col=c2):
            for c in row: c.fill = _fl; c.border = BRD
    if _al: cell.alignment = _al



def build_sheet(ws, rows, col_label, col_key):
    ws.sheet_view.showGridLines = False
    NCOLS = 26

    # Title
    ws.row_dimensions[1].height = 28
    mc(ws, 1, 1, 1, NCOLS,
       f"ПРОГНОЗ vs ФАКТ — {col_label} | Дек 2025 – Фев 2026",
       _fn=fn(True, C_WHT, 13), _fl=fl(C_DARK), _al=CTR)

    ws.row_dimensions[2].height = 14
    mc(ws, 2, 1, 2, NCOLS,
       "Обучение: янв 2023 – ноя 2025 | "
       "Выбросы: IQR × 1.5 → заменяются медианой соседей | "
       "Δ = Факт – Прогноз | Зелёный = модель недооценила, Красный = переоценила",
       _fn=fn(italic=True, color="555555", size=8), _fl=fl("F0F4FA"), _al=CTR)

    # Header rows 4-5
    ws.row_dimensions[4].height = 20
    ws.row_dimensions[5].height = 18

    for c, v in [(1,"№"),(2,"Группа"),(3,"Артикул"),(4,"Номенклатура"),(5,"Тип"),(6,"Аналоги")]:
        mc(ws, 4, c, 5, c, v, _fn=fn(True, C_WHT, 9), _fl=fl(C_DARK), _al=CTR)

    month_colors = ["2C3E50", "1A252F", "0D1B2A"]
    for i, (lbl, bg) in enumerate(zip(FORECAST_LABELS, month_colors)):
        base = 7 + i * 5
        mc(ws, 4, base, 4, base+4, lbl,
           _fn=fn(True, C_WHT, 10), _fl=fl(bg), _al=CTR)
        sub_hdrs = [
            ("Прогноз", C_ORG), ("Без выбр.", C_PRP),
            ("Факт",    C_TL),  ("Δ (осн)",  C_RED),
            ("Δ (чист)",C_GRN),
        ]
        for j, (sh, sc_c) in enumerate(sub_hdrs):
            sc(ws, 5, base+j, sh, _fn=fn(True, C_WHT, 8), _fl=fl(sc_c), _al=CTR)

    mc(ws, 4, 22, 4, 26, "ИТОГО КВАРТАЛ",
       _fn=fn(True, C_WHT, 10), _fl=fl(C_GRN), _al=CTR)
    for j, (sh, sc_c) in enumerate([
        ("Прогноз", C_ORG), ("Без выбр.", C_PRP),
        ("Факт", C_TL), ("Δ (осн)", C_RED), ("Δ (чист)", C_GRN)
    ]):
        sc(ws, 5, 22+j, sh, _fn=fn(True, C_WHT, 8), _fl=fl(sc_c), _al=CTR)

    # Data rows
    for idx, row in enumerate(rows):
        r = 6 + idx
        ws.row_dimensions[r].height = 28
        bg = C_STR if idx % 2 == 0 else C_WHT
        d  = row[col_key]

        sc(ws, r, 1, idx+1,          _fn=fn(True),    _fl=fl(bg), _al=CTR)
        sc(ws, r, 2, row['grp'],      _fn=fn(True),    _fl=fl(bg), _al=CTR)
        sc(ws, r, 3, row['article'],  _fn=fn(size=8),  _fl=fl(bg), _al=CTR)
        sc(ws, r, 4, row['name'],     _fn=fn(size=8),  _fl=fl(bg), _al=LFT)
        sc(ws, r, 5, row['type'],     _fn=fn(size=8),  _fl=fl(bg), _al=CTR)

        # Аналоги + выбросы
        out_note = ""
        if d['outliers']:
            parts = [f"{o['date']}: {o['original']:.0f}→{o['replacement']:.0f}"
                     for o in d['outliers']]
            out_note = " | ⚠ выбросы: " + "; ".join(parts)
        method_note = f" | Метод: {d['method_clean']}"
        zeros_note  = f" | Нули: {d.get('zero_ratio',0)}%"
        sc(ws, r, 6, row['analogs'] + out_note + method_note + zeros_note,
           _fn=fn(size=7), _fl=fl(bg), _al=LFT)

        sum_raw = 0; sum_clean = 0; sum_act = 0
        for i in range(3):
            base    = 7 + i * 5
            fc_raw  = round(d['fc_raw'][i],   1)
            fc_cln  = round(d['fc_clean'][i], 1)
            act_v   = round(d['act'][i],      1)
            d_raw   = round(act_v - fc_raw,   1)
            d_cln   = round(act_v - fc_cln,   1)
            sum_raw   += fc_raw
            sum_clean += fc_cln
            sum_act   += act_v

            sc(ws, r, base,   fc_raw, _fn=fn(True, C_ORG), _fl=fl(C_ORL), _al=RGT, fmt='#,##0.0')
            sc(ws, r, base+1, fc_cln, _fn=fn(True, C_PRP), _fl=fl(C_PRL), _al=RGT, fmt='#,##0.0')
            sc(ws, r, base+2, act_v,  _fn=fn(True, C_TL),  _fl=fl(C_TLL), _al=RGT, fmt='#,##0.0')

            for delta, col_i in [(d_raw, base+3), (d_cln, base+4)]:
                clr = C_TL  if delta >= 0 else C_RED
                bg2 = C_TLL if delta >= 0 else C_RDL
                sc(ws, r, col_i, f"{'+' if delta>=0 else ''}{delta:.1f}",
                   _fn=fn(True, clr), _fl=fl(bg2), _al=CTR)

        # Totals
        td_raw  = round(sum_act - sum_raw,   1)
        td_cln  = round(sum_act - sum_clean, 1)
        sc(ws, r, 22, round(sum_raw,   1), _fn=fn(True, C_ORG), _fl=fl(C_ORL), _al=RGT, fmt='#,##0.0')
        sc(ws, r, 23, round(sum_clean, 1), _fn=fn(True, C_PRP), _fl=fl(C_PRL), _al=RGT, fmt='#,##0.0')
        sc(ws, r, 24, round(sum_act,   1), _fn=fn(True, C_TL),  _fl=fl(C_TLL), _al=RGT, fmt='#,##0.0')
        for delta, col_i in [(td_raw, 25), (td_cln, 26)]:
            clr = C_TL  if delta >= 0 else C_RED
            bg2 = C_TLL if delta >= 0 else C_RDL
            sc(ws, r, col_i, f"{'+' if delta>=0 else ''}{delta:.1f}",
               _fn=fn(True, clr), _fl=fl(bg2), _al=CTR)

    # Grand total row
    tr = 6 + len(rows)
    ws.row_dimensions[tr].height = 20
    mc(ws, tr, 1, tr, 6, "ИТОГО ТОП-10",
       _fn=fn(True, C_WHT, 10), _fl=fl(C_DARK), _al=CTR)

    for i in range(3):
        base = 7 + i * 5
        sf = round(sum(row[col_key]['fc_raw'][i]   for row in rows), 1)
        sc2= round(sum(row[col_key]['fc_clean'][i] for row in rows), 1)
        sa = round(sum(row[col_key]['act'][i]      for row in rows), 1)
        sc(ws, tr, base,   sf,  _fn=fn(True,C_WHT), _fl=fl(C_ORG), _al=RGT, fmt='#,##0.0')
        sc(ws, tr, base+1, sc2, _fn=fn(True,C_WHT), _fl=fl(C_PRP), _al=RGT, fmt='#,##0.0')
        sc(ws, tr, base+2, sa,  _fn=fn(True,C_WHT), _fl=fl(C_TL),  _al=RGT, fmt='#,##0.0')
        for delta, col_i in [(round(sa-sf,1), base+3), (round(sa-sc2,1), base+4)]:
            sc(ws, tr, col_i, f"{'+' if delta>=0 else ''}{delta:.1f}",
               _fn=fn(True,C_WHT), _fl=fl(C_RED if delta<0 else C_TL), _al=CTR)

    tf  = round(sum(sum(row[col_key]['fc_raw'])   for row in rows), 1)
    tc  = round(sum(sum(row[col_key]['fc_clean']) for row in rows), 1)
    ta  = round(sum(sum(row[col_key]['act'])      for row in rows), 1)
    sc(ws,tr,22,tf,_fn=fn(True,C_WHT,10),_fl=fl(C_ORG),_al=RGT,fmt='#,##0.0')
    sc(ws,tr,23,tc,_fn=fn(True,C_WHT,10),_fl=fl(C_PRP),_al=RGT,fmt='#,##0.0')
    sc(ws,tr,24,ta,_fn=fn(True,C_WHT,10),_fl=fl(C_TL), _al=RGT,fmt='#,##0.0')
    for delta, col_i in [(round(ta-tf,1),25),(round(ta-tc,1),26)]:
        sc(ws,tr,col_i,f"{'+' if delta>=0 else ''}{delta:.1f}",
           _fn=fn(True,C_WHT,10),_fl=fl(C_RED if delta<0 else C_TL),_al=CTR)

    # MAPE row
    mr = tr + 1
    ws.row_dimensions[mr].height = 16
    mape_raw  = np.mean([abs(sum(row[col_key]['act'][i] - row[col_key]['fc_raw'][i]
                               for i in range(3)) /
                            (sum(row[col_key]['act']) + 1e-9)) * 100
                         for row in rows])
    mape_cln  = np.mean([abs(sum(row[col_key]['act'][i] - row[col_key]['fc_clean'][i]
                               for i in range(3)) /
                            (sum(row[col_key]['act']) + 1e-9)) * 100
                         for row in rows])
    mc(ws, mr, 1, mr, NCOLS,
       f"MAPE: Прогноз (осн.) = {mape_raw:.1f}%  |  "
       f"Прогноз (без выбросов) = {mape_cln:.1f}%  |  "
       f"{'✅ Очистка выбросов улучшила точность' if mape_cln < mape_raw else '⚠ Очистка не улучшила'}  |  "
       f"Методы: auto-ETS (6 вариантов) + Croston (при >40% нулей)",
       _fn=fn(bold=True, color=C_GRN if mape_cln < mape_raw else C_RED, size=9),
       _fl=fl("FDFEFE"), _al=LFT)

    # Column widths
    ws.column_dimensions['A'].width = 4
    ws.column_dimensions['B'].width = 7
    ws.column_dimensions['C'].width = 13
    ws.column_dimensions['D'].width = 34
    ws.column_dimensions['E'].width = 13
    ws.column_dimensions['F'].width = 40
    for i in range(3):
        for j, w in enumerate([10, 11, 10, 8, 8]):
            ws.column_dimensions[get_column_letter(7 + i*5 + j)].width = w
    for j, w in enumerate([10, 11, 10, 8, 8]):
        ws.column_dimensions[get_column_letter(22 + j)].width = w
    ws.freeze_panes = 'G6'


def build_outlier_sheet(ws, rows_sale, rows_rep):
    """Отдельный лист — подробная таблица всех выбросов."""
    ws.title = "⚠ Выбросы"
    ws.sheet_view.showGridLines = False

    ws.row_dimensions[1].height = 26
    mc(ws, 1, 1, 1, 8,
       "ОБНАРУЖЕННЫЕ ВЫБРОСЫ — автоматическая корректировка (IQR × 1.5)",
       _fn=fn(True, C_WHT, 12), _fl=fl(C_DARK), _al=CTR)

    hdrs = ["Группа", "Номенклатура", "Тип (прод/рем)",
            "Дата", "Исходное значение", "Порог IQR",
            "Заменено на", "Δ корректировка"]
    ws.row_dimensions[3].height = 18
    for ci, h in enumerate(hdrs):
        sc(ws, 3, 1+ci, h, _fn=fn(True, C_WHT, 9), _fl=fl(C_MID), _al=CTR)

    r = 4
    for rows, kind in [(rows_sale, "Продажи"), (rows_rep, "Ремонт")]:
        for row in rows:
            for o in row['sale' if kind == "Продажи" else 'repair']['outliers']:
                ws.row_dimensions[r].height = 16
                delta = round(o['replacement'] - o['original'], 1)
                bg = C_RDL
                sc(ws,r,1,row['grp'],          _fn=fn(True),  _fl=fl(bg),_al=CTR)
                sc(ws,r,2,row['name'][:50],    _fn=fn(size=8),_fl=fl(bg),_al=LFT)
                sc(ws,r,3,kind,                _fn=fn(size=8),_fl=fl(bg),_al=CTR)
                sc(ws,r,4,o['date'],           _fn=fn(True),  _fl=fl(bg),_al=CTR)
                sc(ws,r,5,o['original'],       _fn=fn(True,C_RED),_fl=fl(C_RDL),_al=RGT,fmt='#,##0.0')
                sc(ws,r,6,o['threshold'],      _fn=fn(),      _fl=fl(bg),_al=RGT,fmt='#,##0.0')
                sc(ws,r,7,o['replacement'],    _fn=fn(True,C_GRN),_fl=fl(C_GNL),_al=RGT,fmt='#,##0.0')
                sc(ws,r,8,f"{delta:+.1f}",     _fn=fn(True,C_RED),_fl=fl(bg),_al=CTR)
                r += 1

    for ci, w in enumerate([7,40,12,10,14,12,13,14]):
        ws.column_dimensions[get_column_letter(1+ci)].width = w
    ws.freeze_panes = 'A4'


# ── Точность прогноза (текстовый вывод) ──────────────────────────────────────
def print_accuracy(rows, col_key, label):
    print(f"\n  {'─'*75}")
    print(f"  ТОЧНОСТЬ — {label}")
    print(f"  {'Гр.':>5}  {'Нули':>5}  {'Метод':>20}  {'fc_raw Q3':>10}  "
          f"{'fc_cln Q3':>10}  {'Факт Q3':>9}  {'Δ raw':>7}  {'Δ clean':>8}  MAPE_raw  MAPE_cln")
    print(f"  {'─'*75}")
    for row in rows:
        d = row[col_key]
        fr = round(sum(d['fc_raw']),   1)
        fc = round(sum(d['fc_clean']), 1)
        fa = round(sum(d['act']),      1)
        dr = round(fa - fr, 1)
        dc = round(fa - fc, 1)
        mape_r = abs(dr) / (fa + 1e-9) * 100
        mape_c = abs(dc) / (fa + 1e-9) * 100
        better = "✅" if mape_c < mape_r else "  "
        method = d.get('method_clean', '—')
        zeros  = d.get('zero_ratio', 0)
        print(f"  {row['grp']:>5}  {zeros:>4}%  {method:>20}  {fr:>10.1f}  "
              f"{fc:>10.1f}  {fa:>9.1f}  {dr:>+7.1f}  {dc:>+8.1f}  "
              f"{mape_r:>7.1f}%  {mape_c:>7.1f}% {better}")


# ── MAIN ──────────────────────────────────────────────────────────────────────
def main():
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        script_dir = os.getcwd()

    data_path = os.path.join(script_dir, DATA_FILE)
    if not os.path.exists(data_path):
        print(f"\n❌  Файл не найден: {data_path}")
        sys.exit(1)

    print("\n" + "="*65)
    print("  ПРОГНОЗ vs ФАКТ | Дек 2025 – Фев 2026 | С обработкой выбросов")
    print("="*65)
    print(f"\n  Загрузка {DATA_FILE}...")
    df_train, monthly, group_meta, actual, all_groups = load_data(data_path)
    print(f"  Загружено. Групп: {len(all_groups)}")

    # ── Топ-20 для справки ────────────────────────────────────────────────────
    summary = (df_train.groupby('Номер группы')
               .agg(Продажа=('Продажа', 'sum'),
                    Ремонт=('Ремонт', 'sum'),
                    Номенклатура=('Номенклатура', 'first'))
               .reset_index()
               .sort_values('Продажа', ascending=False))

    print("\n  Топ-20 по продажам (для справки):")
    for _, row in summary.head(20).iterrows():
        print(f"    Гр.{int(row['Номер группы']):4d} | "
              f"Прод: {row['Продажа']:6.0f} | "
              f"Рем: {row['Ремонт']:5.0f} | "
              f"{str(row['Номенклатура'])[:50]}")

    def parse_input(raw):
        nums = set()
        raw = raw.replace(',', ' ')
        for part in raw.split():
            if '-' in part and not part.startswith('-'):
                a, b = part.split('-', 1)
                try: nums.update(range(int(a), int(b) + 1))
                except ValueError: pass
            else:
                try: nums.add(int(part))
                except ValueError: pass
        return sorted(nums)

    # ── Интерактивный цикл ────────────────────────────────────────────────────
    while True:
        print("\n" + "-"*65)
        print("  Форматы: '9 146 205'  |  '9,146'  |  '9-15'  |  'топ10'")
        print("  'поиск <слово>' — поиск по названию  |  'выход' — выйти")
        raw = input("\n  Номера групп: ").strip()

        if raw.lower() in ('выход', 'exit', 'q'):
            print("  До свидания!"); return

        if raw.lower().startswith('поиск'):
            q = raw[5:].strip() or input("  Слово для поиска: ").strip()
            found = summary[summary['Номенклатура'].str.contains(q, case=False, na=False)]
            if found.empty:
                print(f"  Ничего не найдено по: '{q}'")
            else:
                for _, r in found.iterrows():
                    print(f"    Гр.{int(r['Номер группы']):4d} | "
                          f"{r['Продажа']:6.0f} прод. | "
                          f"{str(r['Номенклатура'])[:55]}")
            continue

        if raw.lower() in ('топ10', 'топ', 'top'):
            requested = summary.head(10)['Номер группы'].astype(int).tolist()
        else:
            requested = parse_input(raw)

        if not requested:
            print("  ⚠️  Не удалось распознать номера. Попробуйте ещё раз.")
            continue

        valid   = [g for g in requested if g in all_groups]
        invalid = [g for g in requested if g not in all_groups]
        if invalid:
            print(f"  ⚠️  Не найдены в данных: {invalid}")
        if not valid:
            print("  ❌  Ни одна группа не найдена.")
            continue

        top10_sale = valid
        top10_rep  = valid
        print(f"\n  Выбранные группы: {valid}")

        print("\n  Строим прогноз...")
        all_results = {}
        for grp in list(dict.fromkeys(top10_sale + top10_rep)):
            print(f"    → Группа {grp}...", end='', flush=True)
            r = forecast_group(monthly, group_meta, actual, grp)
            if r:
                all_results[grp] = r
                d = r['sale']
                print(f"  {len(d['outliers'])} выбросов | "
                      f"fc={d['fc_raw']} → clean={d['fc_clean']} | fact={d['act']}")
            else:
                print("  нет данных")

        rows_sale = [all_results[g] for g in top10_sale if g in all_results]
        rows_rep  = [all_results[g] for g in top10_rep  if g in all_results]

        print_accuracy(rows_sale, 'sale',   'ПРОДАЖИ')
        print_accuracy(rows_rep,  'repair', 'РЕМОНТ')

        wb = openpyxl.Workbook()
        ws1 = wb.active; ws1.title = "📈 Продажи"
        build_sheet(ws1, rows_sale, "ПРОДАЖИ", "sale")
        ws2 = wb.create_sheet("🔧 Ремонт")
        build_sheet(ws2, rows_rep, "РЕМОНТ", "repair")
        ws3 = wb.create_sheet()
        build_outlier_sheet(ws3, rows_sale, rows_rep)

        try:
            _sdir = os.path.dirname(os.path.abspath(__file__))
        except NameError:
            _sdir = os.getcwd()
        groups_str = '_'.join(str(g) for g in valid[:5])
        out_path = os.path.join(_sdir, f"forecast_vs_fact_{groups_str}.xlsx")
        wb.save(out_path)
        print(f"\n  ✅  Файл сохранён: {out_path}")
        print("  Листы: 📈 Продажи | 🔧 Ремонт | ⚠ Выбросы")

        again = input("\n  Ещё один прогноз? (да/нет): ").strip().lower()
        if again not in ('да', 'д', 'y', 'yes'):
            print("  До свидания!"); return


if __name__ == "__main__":
    main()


  ПРОГНОЗ vs ФАКТ | Дек 2025 – Фев 2026 | С обработкой выбросов

  Загрузка final.xlsx...
  Загружено. Групп: 2112

  Топ-20 по продажам (для справки):
    Гр.   9 | Прод:   3164 | Рем:   568 | Замок зажигания 90173
    Гр. 223 | Прод:   1762 | Рем:   108 | Датчик угла колес 000004697
    Гр. 116 | Прод:   1544 | Рем:   254 | Фильтр масляный C1049
    Гр. 203 | Прод:   1385 | Рем:   183 | Фильтр гидравлический ST30007
    Гр. 145 | Прод:   1262 | Рем:   383 | Фильтр FF5785
    Гр. 118 | Прод:   1246 | Рем:   232 | Фильтр масляный
    Гр. 113 | Прод:   1192 | Рем:   418 | Фильтр 90031459A
    Гр. 146 | Прод:   1103 | Рем:   410 | Фильтр воздушный Z32540
    Гр. 144 | Прод:   1073 | Рем:   353 | Фильтр топливный
    Гр.   7 | Прод:    959 | Рем:   174 | Кнопка Стоп 122514GT
    Гр. 326 | Прод:    771 | Рем:   140 | Кабель 1020703603
    Гр. 101 | Прод:    655 | Рем:    74 | Фильтр топливный FG533
    Гр.   4 | Прод:    647 | Рем:   104 | Разъем ВПУ (папа)
    Гр. 167 | Прод:    635 | Ре